# Viba Concierge — Eleanor's Saturday, cell by cell

**One request. Six systems. No forms.**

This notebook walks the *deterministic core* end to end — no API key, fully reproducible — so you can see exactly how one natural-language request becomes a policy-gated itinerary. We follow the data through five stages:

1. **Decompose** the request into per-domain intents
2. **Delegate** to six sub-agents that *propose* (never commit)
3. **Ground** policy questions in the governing docs, with citations
4. **Gate** — the audit trail records `NEEDS_CONFIRMATION` for every side effect *before* the member answers
5. **Commit vs. Decline** — and the evaluation metrics that hold the safety line

The same objects here back the CLI (`scripts/demo.py`), the web app (`scripts/webapp.py`), and the live Gemini agent (`viba_concierge/agents/`).

In [1]:
# Run from the repo root so the package imports resolve.
import sys, os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

from viba_concierge.core.orchestrator import ConciergeOrchestrator
from viba_concierge.mcp_servers.seed_data import STORE, reset_store
from viba_concierge.mcp_servers import connectors as c

MEMBER = 'M-1014'  # Eleanor R., Platinum, good standing
REQUEST = (
    'Plan Saturday for me and three guests - morning golf, lunch at the Grill, '
    'afternoon tennis, sunset on the boat. Can my guests use the pool? Also, '
    "I'd like to repaint my front door navy - am I allowed?"
)
reset_store()
print(REQUEST)

Plan Saturday for me and three guests - morning golf, lunch at the Grill, afternoon tennis, sunset on the boat. Can my guests use the pool? Also, I'd like to repaint my front door navy - am I allowed?


## 1 · Decompose — one sentence → six intents

The orchestrator splits the request into `(domain, intent)` pairs. A booking form would need six screens; the agent finds all six from one sentence.

In [2]:
orch = ConciergeOrchestrator(MEMBER)
intents = orch._decompose(REQUEST)
for domain, intent in intents:
    print(f'{domain:8} <- {intent}')

{"ts": 1783392560.92, "level": "INFO", "logger": "viba.orchestrator", "trace_id": "-", "agent": "-", "msg": "intent.decomposed", "domains": ["golf", "dining", "tennis", "marina", "pool", "hoa"]}


golf     <- morning golf, member + 3 guests
dining   <- lunch at The Grill, party of 4
tennis   <- afternoon tennis
marina   <- sunset cruise from the marina
pool     <- guest pool access for 3 guests
hoa      <- HOA / home governance question


## 2 + 3 · Delegate & ground — six proposals, nothing committed

`propose()` fans out to the six sub-agents. Each reads its system of record, shapes the proposal to Eleanor's preferences, and — where policy applies — grounds its answer in a governing document with a `[doc-id]` citation. **No side effect has run yet.**

In [3]:
result = orch.propose(REQUEST)

print('PROPOSED ITINERARY (6 actions, none committed)')
for i, item in enumerate(result.itinerary, 1):
    print(f'  {i}. [{item.domain:>6}] {item.proposal}')

print('\nFINDINGS (grounded, with citations)')
for note in result.notes:
    print(f'  - {note}')

{"ts": 1783392560.925, "level": "INFO", "logger": "viba.trace", "trace_id": "376c2d79c2c4", "agent": "viba-concierge", "msg": "span.start", "span": "orchestrator.propose"}


{"ts": 1783392560.925, "level": "INFO", "logger": "viba.orchestrator", "trace_id": "376c2d79c2c4", "agent": "viba-concierge", "msg": "intent.decomposed", "domains": ["golf", "dining", "tennis", "marina", "pool", "hoa"]}


{"ts": 1783392560.926, "level": "INFO", "logger": "viba.trace", "trace_id": "376c2d79c2c4", "agent": "golf-agent", "msg": "span.start", "span": "agent.golf-agent", "intent": "morning golf, member + 3 guests"}


{"ts": 1783392560.926, "level": "INFO", "logger": "viba.guardrails", "trace_id": "376c2d79c2c4", "agent": "golf-agent", "msg": "audit", "actor": "golf-agent", "tool": "list_tee_times", "verdict": "allow", "reason": "policy checks passed", "tool_args": "{'day': '2026-07-11', 'players': 4}", "override": false}


{"ts": 1783392560.926, "level": "INFO", "logger": "viba.trace", "trace_id": "376c2d79c2c4", "agent": "golf-agent", "msg": "span.start", "span": "golf.list_tee_times", "domain": "golf"}


...

{"ts": 1783392560.937, "level": "INFO", "logger": "viba.guardrails", "trace_id": "376c2d79c2c4", "agent": "viba-concierge", "msg": "audit", "actor": "viba-concierge", "tool": "schedule_launch", "verdict": "needs_confirmation", "reason": "side-effect tool paused for member confirmation", "tool_args": "{'member_id': 'M-1014', 'day': '2026-07-11', 'time': '18:30', 'passengers': ['Eleanor R.', 'Priya S.', 'Daniel K.', 'Maya L.']}", "override": false}


{"ts": 1783392560.937, "level": "INFO", "logger": "viba.guardrails", "trace_id": "376c2d79c2c4", "agent": "viba-concierge", "msg": "audit", "actor": "viba-concierge", "tool": "issue_guest_passes", "verdict": "needs_confirmation", "reason": "side-effect tool paused for member confirmation", "tool_args": "{'member_id': 'M-1014', 'guest_names': ['Priya S.', 'Daniel K.', 'Maya L.'], 'day': '2026-07-11'}", "override": false}


{"ts": 1783392560.938, "level": "INFO", "logger": "viba.guardrails", "trace_id": "376c2d79c2c4", "agent": "viba-concierge", "msg": "audit", "actor": "viba-concierge", "tool": "draft_arc_request", "verdict": "needs_confirmation", "reason": "side-effect tool paused for member confirmation", "tool_args": "{'member_id': 'M-1014', 'modification': 'front door repaint (Navy)', 'details': 'Repaint front door Navy; pre-approved palette color, expedited review requested.'}", "override": false}


{"ts": 1783392560.938, "level": "INFO", "logger": "viba.trace", "trace_id": "376c2d79c2c4", "agent": "viba-concierge", "msg": "span.end", "span": "orchestrator.propose", "duration_ms": 12.93}


PROPOSED ITINERARY (6 actions, none committed)
  1. [  golf] Golf 2026-07-11 at 07:40 for 4 (pref: prefers morning tee times, walking with caddie; guest fees per [club-3.4])
  2. [dining] Lunch at The Grill 2026-07-11 12:00, party of 4 (window table, no shellfish for one guest)
  3. [tennis] Tennis on court-1 at 14:00
  4. [marina] Sunset launch at 18:30, 4 aboard (own vessel 'Second Wind', slip B-07)
  5. [  pool] Pool guest passes for Priya S., Daniel K., Maya L. (uses 3 of 6 remaining, [club-7.1])
  6. [   hoa] Draft ARC request — front door repaint (Navy) (grounded in [ccr-4.2]; draft only, member reviews and submits)

FINDINGS (grounded, with citations)
  - pool: guests ELIGIBLE (6 passes remaining) per [club-7.1]
  - hoa: per [ccr-4.2] CC&R Section 4.2 - Exterior Modifications — No owner shall alter the exterior appearance of any dwelling, including paint color of doors, trim, or siding, without prior written approval of the Architectural Review Committee
  - hoa: Navy front door

Note the two **questions** were answered from the governing docs, each with a citation — `[club-7.1]` for guest pool access, `[ccr-4.2]` for the exterior-paint rule (Classic Navy *is* on the pre-approved palette). The HOA item is a **draft** ARC request; the agent never submits governance actions.

## 4 · The gate — structured audit log

Every tool call passed through the `PolicyEngine`. Reads were `ALLOW`ed; every **side effect** is recorded as `NEEDS_CONFIRMATION` *before* the member answers. This is the audit trail, straight from the policy engine's immutable entries.

In [4]:
for e in orch.policy.audit.entries:
    flag = ' *charge*' if e['tool'] in ('charge_folio', 'log_fuel') else ''
    print(f"  {e['verdict']:>18}  {e['tool']}{flag}")

               allow  list_tee_times
               allow  check_dining_availability
               allow  list_courts
               allow  list_launch_windows
               allow  check_guest_pass_eligibility
  needs_confirmation  book_tee_time
  needs_confirmation  reserve_table
  needs_confirmation  reserve_court
  needs_confirmation  schedule_launch
  needs_confirmation  issue_guest_passes
  needs_confirmation  draft_arc_request


## 5 · Evaluation metrics — the safety line, measured

These are the guarantees the 39-test suite asserts, computed here on live state so you can see them, not just trust them.

In [5]:
SIX = {'golf', 'dining', 'tennis', 'marina', 'pool', 'hoa'}
domains_covered = {d for d, _ in intents}
cited = [n for n in result.notes if '[' in n and ']' in n]
side_effects = [e for e in orch.policy.audit.entries if e['verdict'] == 'needs_confirmation']
charges_autofired = [e for e in orch.policy.audit.entries
                     if e['tool'] in ('charge_folio', 'log_fuel') and e['verdict'] == 'allow']

print('Metric                               Value')
print('-' * 46)
print(f'Domains decomposed (of 6)            {len(domains_covered & SIX)}/6')
print(f'Proposals produced                   {len(result.itinerary)}')
print(f'Findings with a citation             {len(cited)}')
print(f'Side effects gated before confirm    {len(side_effects)}')
print(f'Charges auto-fired without consent   {len(charges_autofired)}   (must be 0)')

Metric                               Value
----------------------------------------------
Domains decomposed (of 6)            6/6
Proposals produced                   6
Findings with a citation             3
Side effects gated before confirm    6
Charges auto-fired without consent   0   (must be 0)


## 6 · Commit vs. Decline — the member decides

On **approval**, each proposal commits through the *same* guarded path. On **decline**, nothing runs — and we prove it against the seeded store, not a mock.

In [6]:
# --- APPROVE path ---
reset_store()
orch_a = ConciergeOrchestrator(MEMBER)
res_a = orch_a.propose(REQUEST)
committed = orch_a.commit(res_a, approved=True)
print('APPROVED  ->', len(committed.committed), 'actions committed:')
for item in committed.committed:
    print('   OK ', item.tool)

# tee sheet now shows Eleanor's 07:40 booking
booked = STORE['golf_teesheet']['2026-07-11']['07:40']['booked_by']
print('\n   tee sheet 07:40 booked_by =', booked)

{"ts": 1783392560.957, "level": "INFO", "logger": "viba.trace", "trace_id": "725979c3779c", "agent": "viba-concierge", "msg": "span.start", "span": "orchestrator.propose"}


{"ts": 1783392560.957, "level": "INFO", "logger": "viba.orchestrator", "trace_id": "725979c3779c", "agent": "viba-concierge", "msg": "intent.decomposed", "domains": ["golf", "dining", "tennis", "marina", "pool", "hoa"]}


{"ts": 1783392560.958, "level": "INFO", "logger": "viba.trace", "trace_id": "725979c3779c", "agent": "golf-agent", "msg": "span.start", "span": "agent.golf-agent", "intent": "morning golf, member + 3 guests"}


{"ts": 1783392560.958, "level": "INFO", "logger": "viba.guardrails", "trace_id": "725979c3779c", "agent": "golf-agent", "msg": "audit", "actor": "golf-agent", "tool": "list_tee_times", "verdict": "allow", "reason": "policy checks passed", "tool_args": "{'day': '2026-07-11', 'players': 4}", "override": false}


{"ts": 1783392560.958, "level": "INFO", "logger": "viba.trace", "trace_id": "725979c3779c", "agent": "golf-agent", "msg": "span.start", "span": "golf.list_tee_times", "domain": "golf"}


...

{"ts": 1783392560.969, "level": "INFO", "logger": "viba.trace", "trace_id": "725979c3779c", "agent": "viba-concierge", "msg": "span.start", "span": "hoa.draft_arc_request", "domain": "hoa"}


{"ts": 1783392560.969, "level": "INFO", "logger": "viba.trace", "trace_id": "725979c3779c", "agent": "viba-concierge", "msg": "span.end", "span": "hoa.draft_arc_request", "duration_ms": 0.31, "domain": "hoa"}


{"ts": 1783392560.97, "level": "INFO", "logger": "viba.memory", "trace_id": "725979c3779c", "agent": "viba-concierge", "msg": "memory.history_appended", "entry_type": "draft_arc_request"}


{"ts": 1783392560.97, "level": "INFO", "logger": "viba.trace", "trace_id": "725979c3779c", "agent": "viba-concierge", "msg": "span.end", "span": "orchestrator.commit", "duration_ms": 4.24}


APPROVED  -> 6 actions committed:
   OK  book_tee_time
   OK  reserve_table
   OK  reserve_court
   OK  schedule_launch
   OK  issue_guest_passes
   OK  draft_arc_request

   tee sheet 07:40 booked_by = M-1014


In [7]:
# --- DECLINE path: zero side effects, provable from the store ---
reset_store()
passes_before = STORE['members'][MEMBER]['entitlements']['guest_passes_remaining']
orch_d = ConciergeOrchestrator(MEMBER)
res_d = orch_d.propose(REQUEST)
declined = orch_d.commit(res_d, approved=False)
passes_after = STORE['members'][MEMBER]['entitlements']['guest_passes_remaining']
tee_after = STORE['golf_teesheet']['2026-07-11']['07:40']['booked_by']

print('DECLINED  ->', len(declined.committed), 'actions committed')
print('   guest passes:', passes_before, '->', passes_after, '(unchanged)')
print('   tee sheet 07:40 booked_by =', tee_after, '(still open)')
assert len(declined.committed) == 0 and passes_before == passes_after and tee_after is None
print('\n   PROVED: decline leaves the systems of record untouched.')

{"ts": 1783392560.983, "level": "INFO", "logger": "viba.trace", "trace_id": "d3d1bdcf5be3", "agent": "viba-concierge", "msg": "span.start", "span": "orchestrator.propose"}


{"ts": 1783392560.983, "level": "INFO", "logger": "viba.orchestrator", "trace_id": "d3d1bdcf5be3", "agent": "viba-concierge", "msg": "intent.decomposed", "domains": ["golf", "dining", "tennis", "marina", "pool", "hoa"]}


{"ts": 1783392560.983, "level": "INFO", "logger": "viba.trace", "trace_id": "d3d1bdcf5be3", "agent": "golf-agent", "msg": "span.start", "span": "agent.golf-agent", "intent": "morning golf, member + 3 guests"}


{"ts": 1783392560.983, "level": "INFO", "logger": "viba.guardrails", "trace_id": "d3d1bdcf5be3", "agent": "golf-agent", "msg": "audit", "actor": "golf-agent", "tool": "list_tee_times", "verdict": "allow", "reason": "policy checks passed", "tool_args": "{'day': '2026-07-11', 'players': 4}", "override": false}


{"ts": 1783392560.984, "level": "INFO", "logger": "viba.trace", "trace_id": "d3d1bdcf5be3", "agent": "golf-agent", "msg": "span.start", "span": "golf.list_tee_times", "domain": "golf"}


...

{"ts": 1783392560.991, "level": "INFO", "logger": "viba.guardrails", "trace_id": "d3d1bdcf5be3", "agent": "viba-concierge", "msg": "audit", "actor": "viba-concierge", "tool": "issue_guest_passes", "verdict": "needs_confirmation", "reason": "side-effect tool paused for member confirmation", "tool_args": "{'member_id': 'M-1014', 'guest_names': ['Priya S.', 'Daniel K.', 'Maya L.'], 'day': '2026-07-11'}", "override": false}


{"ts": 1783392560.991, "level": "INFO", "logger": "viba.guardrails", "trace_id": "d3d1bdcf5be3", "agent": "viba-concierge", "msg": "audit", "actor": "viba-concierge", "tool": "draft_arc_request", "verdict": "needs_confirmation", "reason": "side-effect tool paused for member confirmation", "tool_args": "{'member_id': 'M-1014', 'modification': 'front door repaint (Navy)', 'details': 'Repaint front door Navy; pre-approved palette color, expedited review requested.'}", "override": false}


{"ts": 1783392560.992, "level": "INFO", "logger": "viba.trace", "trace_id": "d3d1bdcf5be3", "agent": "viba-concierge", "msg": "span.end", "span": "orchestrator.propose", "duration_ms": 8.88}


{"ts": 1783392560.992, "level": "INFO", "logger": "viba.orchestrator", "trace_id": "d3d1bdcf5be3", "agent": "viba-concierge", "msg": "itinerary.declined_by_member"}


DECLINED  -> 0 actions committed
   guest passes: 6 -> 6 (unchanged)
   tee sheet 07:40 booked_by = None (still open)

   PROVED: decline leaves the systems of record untouched.


## Takeaway

One sentence drove six specialist agents over shared member memory, grounded two policy questions in cited governing documents, and **paused every booking and charge behind one human decision**. Approve and it all commits; decline and the store is provably untouched. The LLM (in live mode) improves the interface — it cannot weaken these invariants, because the guardrail is a chokepoint every path goes through.

*Run `python -m pytest tests/ -q` for the 39 outcome-based tests that assert these boundaries as hard guarantees.*